# 🔍 RAG Lab — Exploration Notebook
**Day 9 · AI Application Development Bootcamp**

This notebook walks you through the RAG pipeline step by step, letting you **see the output at every stage** before writing a full application.

---

### How to use this notebook
- Run cells **in order** — later cells depend on earlier ones
- Cells marked `# ✏️ YOUR TURN` have gaps for you to fill in
- Cells marked `# 🔬 EXPERIMENT` invite you to change values and observe what happens
- Write your reflections in the **📝 Reflection** markdown cells

### Sections at a glance
| Part | Topic | Type |
|---|---|---|
| A | Guided pipeline walk-through | Run & read |
| B | Parameter experiments | Fill in + reflect |
| C | Open exploration (your own PDF, multi-query, scores) | Open |
| D | **Web-Augmented RAG** — live web search as a retriever | Fill in |
| E | **RAG Evaluation with RAGAS** *(optional)* — measure quality | Fill in |

### Before you start
Make sure your virtual environment is active and `GROQ_API_KEY` is set:
```bash
source .venv/bin/activate          # macOS/Linux
# or
.venv\Scripts\Activate.ps1        # Windows PowerShell

export GROQ_API_KEY="gsk_..."
```
Then launch Jupyter:
```bash
pip install jupyter  # if not already installed
jupyter notebook
```

---
## ⚙️ Setup — Install & Import

In [23]:
# Run this cell first — installs everything needed for the notebook
# (skip if you already ran pip install during the lecture setup)
import sys
!{sys.executable} -m pip install -q \
    groq python-dotenv \
    langchain langchain-community langchain-chroma \
    langchain-huggingface chromadb pypdf sentence-transformers \
    langchain-groq
print("✅ All packages ready")

✅ All packages ready


In [1]:
import os
import shutil
import math
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from .env if present

# Quick check — will print ✅ if the key is available
api_key = os.getenv("GROQ_API_KEY")
if api_key:
    print(f"✅ GROQ_API_KEY found (starts with: {api_key[:8]}...)")
else:
    print("❌ GROQ_API_KEY not found — set it before continuing")
    print("   export GROQ_API_KEY='gsk_your_key_here'   # macOS/Linux")
    print("   $env:GROQ_API_KEY='gsk_your_key_here'    # Windows PS")

✅ GROQ_API_KEY found (starts with: gsk_kZpi...)


---
## 🅐 PART A — Guided Pipeline Walk-Through

We will build the RAG pipeline one stage at a time and inspect the output at each step.  
Place any PDF you have in the same folder as this notebook and update `PDF_PATH` below.  
If you don't have one handy, you can save any Wikipedia article as a PDF from your browser.

### A1 — Step 1: Load the PDF

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# 📌 Change this to your PDF file name
PDF_PATH = "sample.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"📄 Pages loaded: {len(pages)}")
print()
print("--- Metadata of the first page ---")
print(pages[0].metadata)
print()
print("--- First 500 characters of page 1 ---")
print(pages[0].page_content[:500])

C:\Users\tanji\AppData\Local\Temp\ipykernel_34584\1512225248.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


📄 Pages loaded: 6

--- Metadata of the first page ---
{'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Aelion Sunstrider - Complete Character Archive & Lifespan Codex', 'source': 'sample.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}

--- First 500 characters of page 1 ---
AELION SUNSTRIDER (AELION CALEN'HAD)
High Elf Bladesinger Wizard | Folk Hero & Starlight Warden
Race: Sun Elf (Ar-Tel-Quessir)  |  Class: Wizard (School of Bladesinging) 3 / Fighter 1  |  Alignment: Neutral Good  |  Age: 175 Solar Years
(Equivalent Human Age: ~29) 
RAG INDEXING METADATA & ENTITY MAP:
Primary Entities: Aelion Sunstrider, House Calen'had, Archmage Valthor, Elenia Sunstrider, High Forest, Silverleaf Grove, Silverymoon, The Starlight
Wardens, Vulguroth the Black.
Core Themes: Arcane


**What you should see:**
- `pages` is a Python list — one `Document` object per page
- Each Document has a `metadata` dict containing at minimum `page` (0-indexed) and `source` (the filename)
- These metadata fields travel with each chunk all the way to ChromaDB and show up in your citations

> **Why does the page number start at 0?** PyPDFLoader uses 0-based indexing internally. When displaying to users, add 1: `page + 1`.

### A2 — Step 2: Split into Chunks

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(pages)

print(f"✂️  Total chunks created: {len(chunks)}")
print(f"   Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print()
print("--- Sample: chunk #3 ---")
print(f"Metadata : {chunks[2].metadata}")
print(f"Length   : {len(chunks[2].page_content)} chars")
print(f"Content  : {chunks[2].page_content[:400]}")

✂️  Total chunks created: 25
   Average chunk size: 703 chars

--- Sample: chunk #3 ---
Metadata : {'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Aelion Sunstrider - Complete Character Archive & Lifespan Codex', 'source': 'sample.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}
Length   : 729 chars
Content  : state of trance wherein they access collective racial memories of past lives and ancestral lore. 
The conversion formula between Elven Chronological Age (E) and Equivalent Human Developmental Age (H) operates across two distinct
lifecycle phases: 
H = E × 0.20(Phase 1: Physical Youth, 0 ≤ E ≤ 100 years)
H = 20 + (E − 100) × (80 / 650)(Phase 2: Post-Adulthood Reverie Alignment, 100 < E ≤ 750 years)


**Notice:**
- Chunk metadata still carries the original page number and source filename — inherited from the parent Document
- Chunk size varies because `RecursiveCharacterTextSplitter` respects natural break points
- `chunk_size=800` is the **maximum** — most chunks will be somewhat shorter

### A3 — Step 3: Load the Embedding Model

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

print("Loading embedding model... (first run downloads ~80 MB)")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Model loaded")

# Embed a single test sentence and inspect the vector
test_vector = embeddings.embed_query("What is the deadline for submission?")

print(f"\n🔢 Vector dimensions : {len(test_vector)}")
print(f"   First 8 values    : {[round(v, 4) for v in test_vector[:8]]}")
print(f"   Min value         : {min(test_vector):.4f}")
print(f"   Max value         : {max(test_vector):.4f}")

Loading embedding model... (first run downloads ~80 MB)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model loaded

🔢 Vector dimensions : 384
   First 8 values    : [-0.0141, -0.0333, 0.0125, 0.0354, 0.0086, -0.0128, -0.1207, -0.0198]
   Min value         : -0.1540
   Max value         : 0.1974


**What you should see:**
- Vector has exactly **384 dimensions** (that's the output size of MiniLM-L6)
- Values are small floats, both positive and negative
- These 384 numbers together encode the *meaning* of the sentence

### A4 — Step 4: Demonstrate Cosine Similarity

Before storing anything, let's see cosine similarity in action — the mechanism that powers retrieval.

In [5]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    dot   = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x ** 2 for x in a))
    mag_b = math.sqrt(sum(x ** 2 for x in b))
    return dot / (mag_a * mag_b)

sentences = {
    "query"    : "What is the late submission policy?",
    "similar"  : "Penalty for assignments handed in after the deadline",
    "related"  : "Students must attend all classes",
    "unrelated": "The best recipe for banana bread",
}

vectors   = {k: embeddings.embed_query(v) for k, v in sentences.items()}
query_vec = vectors["query"]

print(f'Base query: "{sentences["query"]}"\n')
for key in ["similar", "related", "unrelated"]:
    score = cosine_similarity(query_vec, vectors[key])
    bar   = "█" * int(score * 30)
    print(f"{score:.4f}  {bar}")
    print(f'         → "{sentences[key]}"')
    print()

Base query: "What is the late submission policy?"

0.4022  ████████████
         → "Penalty for assignments handed in after the deadline"

0.0741  ██
         → "Students must attend all classes"

-0.0138  
         → "The best recipe for banana bread"



**Expected pattern:** `similar` should score highest (different words, same meaning), `related` moderate, `unrelated` very low.  
This is exactly what ChromaDB does for every query — computed across all stored chunk vectors simultaneously.

### A5 — Step 5: Store in ChromaDB

In [6]:
from langchain_chroma import Chroma

CHROMA_DIR = "./chroma_lab_notebook"

if Path(CHROMA_DIR).exists():
    shutil.rmtree(CHROMA_DIR)
    print("🗑  Cleared old index")

print(f"Embedding {len(chunks)} chunks and storing in ChromaDB...")
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name="lab_exploration",
)

count = vector_store._collection.count()
print(f"✅ Done — {count} vectors stored in ChromaDB")
print(f"   Index saved to: {CHROMA_DIR}/")

Embedding 25 chunks and storing in ChromaDB...
✅ Done — 25 vectors stored in ChromaDB
   Index saved to: ./chroma_lab_notebook/


### A6 — Step 6: Retrieve & Inspect Results

In [7]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

QUESTION = "What are the main topics covered in this document?"

retrieved_docs = retriever.invoke(QUESTION)

print(f"❓ Question: {QUESTION}")
print(f"📋 Retrieved {len(retrieved_docs)} chunks:\n")

for i, doc in enumerate(retrieved_docs, 1):
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    print(f"--- Chunk {i} (page {page_display}) ---")
    print(doc.page_content[:300])
    print()

❓ Question: What are the main topics covered in this document?
📋 Retrieved 4 chunks:

--- Chunk 1 (page 4) ---
extraplanar exploitation. 
D&D 5e RAG Knowledge Base — Aelion Sunstrider (Sun Elf Bladesinger) Page 4 of 6

--- Chunk 2 (page 5) ---
base AC.
Codex of Starlight
Cadence Spellbook 3 lbs Bound in dragon-scale leather. Contains 12 First-Level and 4 Second-Level spells
in elven shorthand.
Ring of Selûne's GraceRing (Uncommon) — Heirloom from his mother. Glows softly when fiends or undead approach within 60
feet.
Explorer's Arcane Pac

--- Chunk 3 (page 5) ---
1st Level (4
Slots)
Shield, Absorb Elements, Mage Armor, Magic
Missile, Find Familiar
Includes: Detect Magic (R), Feather Fall, Shield, Absorb Elements, Mage Armor,
Magic Missile, Silvery Barbs, Sleep.
2nd Level (2
Slots) Misty Step, Shadow Blade Includes: Misty Step, Shadow Blade, Mirror Image, Hol

--- Chunk 4 (page 5) ---
PROFICIENCY & SKILL BREAKDOWN
SKILL NAME MOD PROFICIENT?
Acrobatics (DEX) +6 Yes
Arcana (INT) +6 Yes
H

**🔑 This is the most important debugging step in RAG.** Before looking at the LLM's answer, always inspect what the retriever found. If these chunks don't contain the answer, the LLM won't be able to answer correctly — no prompt tuning will fix a retrieval problem.

### A7 — Step 7: Generate an Answer with Groq

In [8]:
from groq import Groq

SYSTEM_PROMPT = """You are a document-based AI assistant.
Answer ONLY using the retrieved context provided below.
For every fact, include the page number in parentheses, e.g. (page 2).
If the answer is not in the context, say: I could not find that in the provided document."""

def format_context(docs):
    parts = []
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get('page', '?')
        page_display = page + 1 if isinstance(page, int) else page
        parts.append(f"[Source {i} | page {page_display}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)

def ask_rag(question, retriever, model="llama-3.1-8b-instant", temperature=0.2):
    """Full RAG loop: retrieve → build prompt → generate → return (answer, docs)."""
    docs    = retriever.invoke(question)
    context = format_context(docs)
    prompt  = f"""Question: {question}

Retrieved context:
{context}

Answer using ONLY the context above. Include page references."""

    client   = Groq(api_key=os.getenv("GROQ_API_KEY"))
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        temperature=temperature,
    )
    return response.choices[0].message.content, docs

answer, retrieved = ask_rag(QUESTION, retriever)

print(f"❓ Question: {QUESTION}")
print()
print("📋 Retrieved chunks:")
for i, doc in enumerate(retrieved, 1):
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    print(f"  [{i}] page {page_display}: {doc.page_content[:80]}...")
print()
print("🤖 Answer:")
print(answer)

❓ Question: What are the main topics covered in this document?

📋 Retrieved chunks:
  [1] page 4: extraplanar exploitation. 
D&D 5e RAG Knowledge Base — Aelion Sunstrider (Sun El...
  [2] page 5: base AC.
Codex of Starlight
Cadence Spellbook 3 lbs Bound in dragon-scale leathe...
  [3] page 5: 1st Level (4
Slots)
Shield, Absorb Elements, Mage Armor, Magic
Missile, Find Fam...
  [4] page 5: PROFICIENCY & SKILL BREAKDOWN
SKILL NAME MOD PROFICIENT?
Acrobatics (DEX) +6 Yes...

🤖 Answer:
The main topics covered in this document are:

1. Aelion Sunstrider's character information, including his spells and equipment, (Source 1, page 4; Source 2, page 5; Source 3, page 5; Source 4, page 5)
2. Aelion Sunstrider's spellbook inscriptions, including his prepared spells and recorded spells in the Codex of Starlight (Source 2, page 5)
3. Aelion Sunstrider's proficiency and skill breakdown, including his proficiency in various skills and his proficiency bonus (Source 4, page 5)
4. Aelion Sunstrider's e

---
## 🅑 PART B — Experiments

Now you modify parameters and observe how they affect retrieval quality. **This is the core skill of RAG engineering.**

### B1 — ✏️ YOUR TURN: Chunk Size Experiment

Re-index your document with **three different chunk sizes** and compare how many chunks are created and how the retrieved text looks.  
Fill in the `build_index()` function below — it should work for all three sizes.

In [9]:
def build_index(pages, chunk_size, chunk_overlap, chroma_dir):
    """
    Build a ChromaDB index from pages with the given chunking parameters.
    Returns (vector_store, chunks)
    """
    # ✏️ YOUR TURN: create the splitter
    # Hint: RecursiveCharacterTextSplitter(chunk_size=..., chunk_overlap=..., separators=[...])
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    # ✏️ YOUR TURN: split the pages into chunks
    chunks = splitter.split_documents(pages)

    if Path(chroma_dir).exists():
        shutil.rmtree(chroma_dir)

    # ✏️ YOUR TURN: create the ChromaDB vector store
    # Hint: Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=chroma_dir, collection_name="exp")
    store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=chroma_dir,
        collection_name="exp",
    )

    return store, chunks


# ── Run for three chunk sizes ─────────────────────────────────────────────────
test_question = "What are the main topics covered in this document?"
results = {}

for size in [300, 800, 1500]:
    overlap = int(size * 0.15)
    store, cks = build_index(pages, size, overlap, f"./chroma_exp_{size}")
    ret  = store.as_retriever(search_kwargs={"k": 3})
    docs = ret.invoke(test_question)
    results[size] = {"chunk_count": len(cks), "retrieved": docs}
    print(f"chunk_size={size:>4}: {len(cks):>4} chunks total | retrieved {len(docs)} docs")

print("\n✅ Experiment complete — run the next cell to compare retrieved text")

chunk_size= 300:   69 chunks total | retrieved 3 docs
chunk_size= 800:   25 chunks total | retrieved 3 docs
chunk_size=1500:   15 chunks total | retrieved 3 docs

✅ Experiment complete — run the next cell to compare retrieved text


In [10]:
# Inspect retrieved chunks for each size side-by-side
for size, data in results.items():
    print(f"\n{'='*60}")
    print(f"  chunk_size = {size}  ({data['chunk_count']} total chunks)")
    print(f"{'='*60}")
    for i, doc in enumerate(data["retrieved"], 1):
        page = doc.metadata.get('page', '?')
        page_display = page + 1 if isinstance(page, int) else page
        print(f"  [{i}] page {page_display} | {len(doc.page_content)} chars")
        print(f"      {doc.page_content[:200]}")
        print()


  chunk_size = 300  (69 total chunks)
  [1] page 1 | 91 chars
      leadership.
D&D 5e RAG Knowledge Base — Aelion Sunstrider (Sun Elf Bladesinger) Page 1 of 6

  [2] page 2 | 292 chars
      Fundamentals
Martial Trial & Exile111 – 140 Years Captain Lyra ShadowstepDexterity, ConstitutionFall of Silverleaf Grove, Guerrilla
Warfare
Wandering Starlight
Warden 141 – 175 Years (Current)Self / W

  [3] page 4 | 251 chars
      CHRONOLOGICAL ERAS & MAJOR LIFE PHASES
V. Feats, Abilities & Martial Development
FEATURE /
TRAIT NAME
SOURCE
ORIGIN GAME MECHANICS & EFFECT IN-LORE NARRATIVE EXPLANATION
Bladesong Wizard
(Bladesinging


  chunk_size = 800  (25 total chunks)
  [1] page 4 | 106 chars
      extraplanar exploitation. 
D&D 5e RAG Knowledge Base — Aelion Sunstrider (Sun Elf Bladesinger) Page 4 of 6

  [2] page 5 | 764 chars
      base AC.
Codex of Starlight
Cadence Spellbook 3 lbs Bound in dragon-scale leather. Contains 12 First-Level and 4 Second-Level spells
in elven shorthand.
Ring of Se

### 📝 Reflection B1

1. **How did chunk count change** as chunk_size increased from 300 → 800 → 1500?

   *Your answer:* The number of total chunks decrease as the chunk_size increased

2. **Which chunk size returned the most useful context** for your question? Why?

   *Your answer:* 800. The chunk contain just the relevant data without cutting anything out and without any fluff.

3. **With chunk_size=300**, did any retrieved chunk seem too short to be useful on its own?

   *Your answer:* Yes. The first chunk basically only contain metadata and the other chunks gets cut off omitting crucial data.

4. **With chunk_size=1500**, did the retrieved chunk contain irrelevant sentences alongside the relevant ones?

   *Your answer:* Yes, the chunks contain the entire table worth of data and it also combines some of the information together adding some unrelated details into a single unit.

### B2 — ✏️ YOUR TURN: The k Parameter Experiment

Using a `chunk_size=800` index, compare what happens when you retrieve **k=1**, **k=4**, and **k=10** chunks.

In [11]:
store_800, _ = build_index(pages, 800, 120, "./chroma_k_exp")

# ✏️ YOUR TURN: pick a question relevant to YOUR document
question = "Summarize Aelion's major life phases and his personal goals."

print(f"Question: {question}\n")

for k in [1, 4, 10]:
    # ✏️ YOUR TURN: create a retriever with this k value and retrieve docs
    # Hint: store_800.as_retriever(search_kwargs={"k": k})
    retriever_k = store_800.as_retriever(search_kwargs={"k": k})
    docs_k      = retriever_k.invoke(question)

    total_chars = sum(len(d.page_content) for d in docs_k)
    pages_found = [d.metadata.get('page', '?') for d in docs_k]
    print(f"k={k:2d} → {len(docs_k)} chunks | ~{total_chars} chars of context | pages: {pages_found}")

Question: Summarize Aelion's major life phases and his personal goals.

k= 1 → 1 chunks | ~788 chars of context | pages: [1]
k= 4 → 4 chunks | ~3091 chars of context | pages: [1, 3, 2, 1]
k=10 → 10 chunks | ~7567 chars of context | pages: [1, 3, 2, 1, 4, 1, 3, 1, 0, 1]


In [12]:
# Compare the ANSWERS generated with k=1 vs k=4 vs k=10
for k in [1, 4, 10]:
    ret_k    = store_800.as_retriever(search_kwargs={"k": k})
    ans_k, _ = ask_rag(question, ret_k)
    print(f"\n{'─'*60}")
    print(f"  k = {k}")
    print(f"{'─'*60}")
    print(ans_k)


────────────────────────────────────────────────────────────
  k = 1
────────────────────────────────────────────────────────────
Based on the provided context, I can summarize Aelion's major life phases and personal goals as follows:

Aelion's major life phases include:

1. Early years (page 2): Aelion exhibited a dual fascination with arcane weave manipulation and martial bladeplay from his earliest years.
2. Traditional Reverie instruction (page 2): Aelion underwent traditional Reverie instruction, learning to navigate the racial memories of his ancestors during trance and mastering various languages.

As for Aelion's personal goals, the context does not explicitly mention them. However, it can be inferred that Aelion's fascination with arcane weave manipulation and martial bladeplay may indicate a desire to master these skills and potentially become proficient in both areas.

────────────────────────────────────────────────────────────
  k = 4
─────────────────────────────────────

### 📝 Reflection B2

1. **k=1 vs k=4:** Did increasing k improve the answer? In what way?

   *Your answer:* Increasing to 4 provided a much richer timeline, but it still missed the character's explicit goals.

2. **k=10:** Better or worse than k=4? Any sign of context dilution?

   *Your answer:* 10 was better because it successfully retrieved the missing goals from Page 5 without introducing noise.

3. **If you had to pick one k value**, which would you choose and why?

   *Your answer:* 10. It ensure that all of the facts scattered across the document are captured.

### B3 — ✏️ YOUR TURN: Temperature Experiment

In [13]:
retriever_b3 = store_800.as_retriever(search_kwargs={"k": 4})
question_b3  = question  # reuse your question from B2

for temp in [0.0, 0.5, 1.0]:
    # ✏️ YOUR TURN: call ask_rag with the current temperature
    # Hint: ask_rag(question, retriever, temperature=temp)
    answer_t, _ = ask_rag(question_b3, retriever_b3, temperature=temp)

    print(f"\n{'─'*60}")
    print(f"  temperature = {temp}")
    print(f"{'─'*60}")
    print(answer_t)


────────────────────────────────────────────────────────────
  temperature = 0.0
────────────────────────────────────────────────────────────
Aelion's major life phases can be summarized as follows:

1. **Early Life (0-50 DR)**: Aelion was born into a family of High Elven nobility, with his father serving as a High Warden and his mother as a master archivist. He exhibited a dual fascination with arcane weave manipulation and martial bladeplay from an early age (Source 1, page 2).
2. **Reverie Instruction (0-50 DR)**: Aelion underwent traditional Reverie instruction, learning to navigate the racial memories of his ancestors and mastering various languages (Source 1, page 2).
3. **The Last Stand at Sun-Heart (1440 DR)**: Aelion participated in a pivotal battle, serving as an Arcane Rearguard and utilizing spells such as Absorb Elements and Mirror Image. The loss of his ancestral home profoundly altered his worldview (Source 2, page 4).
4. **Exile and Shadow War (1440-1470 DR)**: Aelion 

### 📝 Reflection B3

1. **temperature=0.0 vs 1.0:** How did the answer style differ? Was it more or less consistent on re-run?

   *Your answer:* At 0 the answer provided is more structed and consistent, while at 1 the style is more conversational and creative leading to a higher output variability.

2. **For a factual document Q&A system**, which temperature range would you recommend and why?

   *Your answer:* A lower temperature range is preferred to ensure factual answer and minimal hallucination.

### B4 — 🔬 EXPERIMENT: Ask an Out-of-Scope Question

In [14]:
retriever_b4   = store_800.as_retriever(search_kwargs={"k": 4})
out_of_scope_q = "What is the current price of Bitcoin in Singapore dollars?"

answer_oos, docs_oos = ask_rag(out_of_scope_q, retriever_b4)

print(f"❓ Question: {out_of_scope_q}")
print()
print("📋 What the retriever found (likely unrelated):")
for doc in docs_oos:
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    print(f"  page {page_display}: {doc.page_content[:100]}...")
print()
print("🤖 Answer:")
print(answer_oos)

❓ Question: What is the current price of Bitcoin in Singapore dollars?

📋 What the retriever found (likely unrelated):
  page 3: During this era, Aelion authored his first spellbook—the  Codex of Starlight Cadence—binding it in s...
  page 5: 1st Level (4
Slots)
Shield, Absorb Elements, Mage Armor, Magic
Missile, Find Familiar
Includes: Dete...
  page 6: COMPREHENSIVE RAG KNOWLEDGE BASE — QUICK LOOKUP DIRECTORY
This structured table provides high-densit...
  page 4: BATTLE /
ENGAGEMENT
YEAR
(DR)
AELION'S TACTICAL
ROLE SPELLS UTILIZED OUTCOME & HISTORICAL
SIGNIFICAN...

🤖 Answer:
I could not find the current price of Bitcoin in Singapore dollars in the provided document.


### 📝 Reflection B4

1. Did the model correctly refuse, or did it hallucinate using irrelevant context?

   *Your answer:* It correctly refuse.

2. If it hallucinated: what in `SYSTEM_PROMPT` could you change to make refusal more reliable?

   *Your answer:* 

3. Modify `SYSTEM_PROMPT` in cell A7 to make refusal more explicit, re-run, and report what changed:

   *Your answer:*

---
## 🅒 PART C — Open Exploration

### C1 — ✏️ YOUR TURN: Test Your Own PDF

In [19]:
# ✏️ YOUR TURN: load and index your own PDF, then ask 3 questions
MY_PDF = "sample.pdf"

# YOUR CODE HERE — follow the same steps as Parts A1 → A6
# Step 1: load   →  PyPDFLoader(MY_PDF).load()
# Step 2: chunk  →  build_index() from B1
# Step 3: ask 3 questions with ask_rag()

loader = PyPDFLoader(MY_PDF)
pages = loader.load()

vector_store, chunks = build_index(pages, 800, 120, "./chroma_my_pdf2")

answer_1, retrieved_docs_1 = ask_rag("What is the primary topic or main objective of this document?", retriever)
answer_2, retrieved_docs_2 = ask_rag("What are the key facts, rules, or requirements mentioned?", retriever)
answer_3, retrieved_docs_3 = ask_rag("Summarize any notable conclusions or personal/character details.", retriever)

print("=== Answer 1 ===")
print(answer_1)

print("\n=== Answer 2 ===")
print(answer_2)

print("\n=== Answer 3 ===")
print(answer_3)

=== Answer 1 ===
The primary topic or main objective of this document appears to be the character profile and background information of Aelion Sunstrider, a Sun Elf Bladesinger (Source 1, page 4). 

However, a more specific main objective can be inferred from the "VIII. Present Campaign Hook & The Call to Adventure" section (Source 3, page 5). It seems that the main objective of this document is to provide a campaign hook for a Dungeons & Dragons (D&D) game, specifically involving Aelion Sunstrider's arrival in Waterdeep to investigate unusual planar anomalies and the resurfacing of the Sun-Heart Crystal shards.

=== Answer 2 ===
Based on the provided context, here are the key facts, rules, or requirements mentioned:

1. Aelion Sunstrider's proficiency in various skills is listed, including Acrobatics, Arcana, History, Insight, Perception, Stealth, Nature, and Survival (Source 1, page 5).
2. When Bladesong is active, Aelion Sunstrider gains a +4 bonus to all Acrobatics checks and Conce

### 📝 Reflection C1

| Question | Retrieval correct? | Answer quality (1–5) | Notes |
|---|---|---|---|
| Q1: What is the primary topic or main objective of this document? | Yes  | 5 | Retrieved chunks accurately captured high-level summary and document scope. |
| Q2: What are the key facts, rules, or requirements mentioned? | Yes  | 5 | k=4 pulled sufficient specific rules/data points without missing essential details. |
| Q3: Summarize any notable conclusions or personal/character details. | Yes  | 4 | Retrieved core facts cleanly, though synthesising scattered details across chunks slightly varied depth. |

**Best chunk_size and k for your document?** *Your answer:* chunk_size = 800 and k = 4 provided the ideal balance

### C2 — 🔬 EXPERIMENT: Multi-Query Retrieval

In [26]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY"),
)

base_retriever  = store_800.as_retriever(search_kwargs={"k": 4})
multi_retriever = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)

test_q        = question  # reuse your question from B2
docs_standard = base_retriever.invoke(test_q)
docs_multi    = multi_retriever.invoke(test_q)

print(f"Standard retrieval : {len(docs_standard)} chunks")
print(f"Multi-query        : {len(docs_multi)} chunks (after dedup)")

standard_texts = {d.page_content for d in docs_standard}
extra_chunks   = [d for d in docs_multi if d.page_content not in standard_texts]
print(f"\nExtra chunks found ONLY by multi-query: {len(extra_chunks)}")
for doc in extra_chunks:
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    print(f"  [page {page_display}] {doc.page_content[:150]}")

Standard retrieval : 4 chunks
Multi-query        : 10 chunks (after dedup)

Extra chunks found ONLY by multi-query: 6
  [page 4] extraplanar exploitation. 
D&D 5e RAG Knowledge Base — Aelion Sunstrider (Sun Elf Bladesinger) Page 4 of 6
  [page 5] base AC.
Codex of Starlight
Cadence Spellbook 3 lbs Bound in dragon-scale leather. Contains 12 First-Level and 4 Second-Level spells
in elven shorthan
  [page 5] 1st Level (4
Slots)
Shield, Absorb Elements, Mage Armor, Magic
Missile, Find Familiar
Includes: Detect Magic (R), Feather Fall, Shield, Absorb Element
  [page 1] Relics, Biological Aging Ratio.
Key Attributes: High Intelligence (18), High Dexterity (19), Bladesong Active Ability, Fey Step Teleportation, Keen Se
  [page 2] LIFECYCLE PHASECHRONOLOGICAL
SPAN PRIMARY MENTORCORE ATTRIBUTE
FOCUS KEY MILESTONES ACHIEVED
Childhood (Fey
Infancy) 0 – 25 Years Lady Aredhel Calen'h
  [page 4] spellcasting.
Forged through intense close-quarters combat against
drow shadow-blades.
Fighting Style:
Du

### C3 — 🔬 EXPERIMENT: Retrieval with Similarity Scores

In [27]:
results_with_scores = vector_store.similarity_search_with_score(question, k=6)

print(f"Question: {question}\n")
print(f"{'Similarity':>10}  {'Page':>5}  Content preview")
print("-" * 72)
for doc, score in results_with_scores:
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    similarity   = 1 / (1 + score)  # convert L2 distance to rough similarity
    print(f"{similarity:>10.4f}  {str(page_display):>5}  {doc.page_content[:60]}...")

Question: Summarize Aelion's major life phases and his personal goals.

Similarity   Page  Content preview
------------------------------------------------------------------------
    0.4845      2  ancient ironwood trees. His father,  Lord Eldrin Calen'had, ...
    0.4737      4  BATTLE /
ENGAGEMENT
YEAR
(DR)
AELION'S TACTICAL
ROLE SPELLS ...
    0.4733      3  During this era, Aelion authored his first spellbook—the  Co...
    0.4722      2  Age 50 6' 1" / 135 lbs 30 ft 12 (DEX +2) 1st Level Arcane In...
    0.4597      5  in Undermountain. 
Now standing at the precipice of his grea...
    0.4483      2  LIFECYCLE PHASECHRONOLOGICAL
SPAN PRIMARY MENTORCORE ATTRIBU...


### 📝 Reflection C — Overall

1. Is there a clear similarity score gap between the most and least relevant chunks?

   *Your answer:* Yes, there is a narrow similarity gap between the top result and the bottom result, indicating that many chunks share overlapping terms even if they don't contain the exact answers.

2. **Two most important RAG parameters to tune** for a new document collection?

   *Your answer:* Chunk size and k

3. **Which mini-project did you choose, and why?** What do you expect to be hardest?

   *Your answer:* 

---
## 🌐 PART D — Web-Augmented RAG

### What is Web-Augmented RAG?

So far, every answer has been grounded in a **local PDF** that you indexed yourself. But what if the user asks something that is not in any of your documents — for example, a current event, a recent product release, or a live regulation?

**Web-Augmented RAG** solves this by replacing (or combining) the vector-store retriever with a **live web search**. Instead of fetching chunks from ChromaDB, the system searches the web in real time, retrieves the top results, and uses those as the LLM's context.

```
Standard RAG:          Question → ChromaDB → chunks → LLM → answer
Web-Augmented RAG:     Question → Web search → live results → LLM → answer
Combined (best):       Question → ChromaDB + Web search → merge → LLM → answer
```

### Tool we use: Tavily Search

**Tavily** is a search API designed specifically for LLM applications. It returns clean, structured, LLM-friendly text from web results — not raw HTML. LangChain has a native `TavilySearchResults` integration.

**Free tier:** 1,000 searches/month — more than enough for this lab.  
**Sign up:** https://tavily.com → get your API key in 30 seconds.

### D0 — Install & Set Your Tavily Key

In [28]:
# Install the Tavily integration
import sys
!{sys.executable} -m pip install -q tavily-python langchain-community
print("✅ Tavily installed")

✅ Tavily installed


In [32]:
# Set your Tavily API key
# Get it from: https://app.tavily.com/home  (free, takes ~30 seconds)
#
# Option A — add to your .env file:    TAVILY_API_KEY=tvly-...
# Option B — set it here for this session only (do not commit this notebook with the key visible):
#
# import os
# os.environ["TAVILY_API_KEY"] = "tvly-your-key-here"
load_dotenv()

tavily_key = os.getenv("TAVILY_API_KEY") #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
if tavily_key:
    print(f"✅ TAVILY_API_KEY found (starts with: {tavily_key[:8]}...)")
else:
    print("❌ TAVILY_API_KEY not set.")
    print("   Get a free key at https://app.tavily.com/home")
    print("   Then add TAVILY_API_KEY=tvly-... to your .env file and re-run this cell.")

✅ TAVILY_API_KEY found (starts with: tvly-dev...)


### D1 — Guided: Web Search as a Retriever

Here we use `TavilySearchResults` as a drop-in retriever. The API returns a list of result objects, each with `content` (the page text) and `url`. We convert these into `Document` objects so the rest of our pipeline stays identical — the same `format_context()` and `ask_rag()` functions work unchanged.

In [31]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.documents import Document

# Create the web search tool
# max_results: how many web pages to retrieve per query (3–5 is a good range)
web_search = TavilySearchResults(
    max_results=4,
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
)

def web_search_to_docs(query: str) -> list:
    """
    Run a web search and convert results to Document objects
    so they work with our existing format_context() function.
    Each Document gets:
      - page_content: the retrieved text snippet
      - metadata["source"]: the URL
      - metadata["page"]: "web" (instead of a page number)
    """
    results = web_search.invoke(query)
    docs = []
    for r in results:
        docs.append(Document(
            page_content=r["content"],
            metadata={"source": r["url"], "page": "web"},
        ))
    return docs

# Test with a query that definitely requires current/live information
web_query = "Latest developments in AI regulation in Singapore 2025"
web_docs  = web_search_to_docs(web_query)

print(f"🌐 Web search: '{web_query}'")
print(f"   Retrieved {len(web_docs)} results\n")
for i, doc in enumerate(web_docs, 1):
    print(f"  [{i}] Source: {doc.metadata['source']}")
    print(f"       {doc.page_content[:200]}")
    print()

C:\Users\tanji\AppData\Local\Temp\ipykernel_34584\542086635.py:6: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search = TavilySearchResults(


🌐 Web search: 'Latest developments in AI regulation in Singapore 2025'
   Retrieved 4 results

  [1] Source: https://regulations.ai/regulations/RAI-SG-NA-SUMMARY-2026
       Singapore's AI regulatory landscape is dynamic and continuously evolving, with several key future developments anticipated. The Online Safety (Relief and Accountability) Bill (OSRA Bill) is currently 

  [2] Source: https://digital.nemko.com/regulations/singapore-ai-regulation
       ## 

## High-Impact and Generative AI Oversight

Singapore’s regulatory focus in 2025 extends to the governance of high-impact and generative AI systems, emphasizing robust safety testing, accountabil

  [3] Source: https://www.linkedin.com/posts/nicholasker_ai-regulations-in-2025-is-singapore-getting-activity-7310812823066984448-w7Hz
       AI Regulations in 2025: Is Singapore getting it right? AI is transforming industries at lightning speed, and with that comes a wave of evolving regulations. Hyperight’s latest article lays out six 

### D2 — Guided: Generate an Answer from Web Results

We can now use the same `format_context()` and the same Groq call — just swapping the source of the context from ChromaDB chunks to live web results.

In [33]:
WEB_SYSTEM_PROMPT = """You are a research assistant answering questions using live web search results.
Answer using ONLY the retrieved web content provided below.
For every fact, cite the source URL in parentheses.
If the information is not in the provided results, say: I could not find that in the retrieved web content.
Do NOT use your own training knowledge."""

def ask_web_rag(question: str, max_results: int = 4) -> str:
    """Full Web-RAG loop: web search → format context → Groq → answer."""
    # Step 1: retrieve from the web
    docs    = web_search_to_docs(question)
    context = format_context(docs)  # reusing the same function from Part A

    # Step 2: build prompt and generate
    prompt = f"""Question: {question}

Retrieved web content:
{context}

Answer using ONLY the retrieved content above. Cite the source URL for every claim."""

    client   = Groq(api_key=os.getenv("GROQ_API_KEY"))
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": WEB_SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

# Ask something that requires live, current information
live_question = "What are the latest AI safety regulations announced in 2025?"

print(f"❓ Question: {live_question}")
print()
answer_web = ask_web_rag(live_question)
print("🤖 Answer (grounded in live web results):")
print(answer_web)

❓ Question: What are the latest AI safety regulations announced in 2025?

🤖 Answer (grounded in live web results):
I could not find any information on the latest AI safety regulations announced in 2025 in the retrieved web content.


### D3 — ✏️ YOUR TURN: Compare Local RAG vs Web RAG on the Same Question

Now ask the **same question** to both your local ChromaDB retriever and the web search retriever.  
Choose a question where your local PDF might have partial information but the web would have more current details —  
for example: a topic your PDF covers, but the web has newer developments on.

**Goal:** understand when to use each approach, and when to combine them.

In [40]:
# ✏️ YOUR TURN: pick a question that makes sense for both your local PDF AND the web
# Good examples:
#   - A topic your PDF covers historically, but the web has 2025 updates on
#   - A regulation your PDF mentions, but whose current status you want to verify
#   - A technology your PDF explains, but recent benchmarks exist online

comparison_question = "What magic items does Aelion own in the document, and what are their official 5e mechanics?"

# ── Local RAG (your PDF) ──────────────────────────────────────────────────────
# ✏️ YOUR TURN: use ask_rag() with store_800's retriever
local_retriever = store_800.as_retriever()
local_answer, local_docs = ask_rag(comparison_question, local_retriever)

print("=" * 60)
print("📄 LOCAL RAG (from your PDF)")
print("=" * 60)
print(f"""Sources used: {[f'page {d.metadata.get("page") + 1 if isinstance(d.metadata.get("page"), int) else "?"}' for d in local_docs]}""")
print(local_answer)

print()
print("=" * 60)
print("🌐 WEB RAG (live web search)")
print("=" * 60)

# ✏️ YOUR TURN: use ask_web_rag() with the same question
web_answer = ask_web_rag(comparison_question)
print(web_answer)

📄 LOCAL RAG (from your PDF)
Sources used: ['page 5', 'page 4', 'page 5', 'page 3']
Based on the provided context, Aelion owns the following magic items:

1. Codex of Starlight (page 5) - a spellbook containing 12 First-Level and 4 Second-Level spells in elven shorthand.
2. Ring of Selûne's Grace (page 5) - a ring that glows softly when fiends or undead approach within 60 feet.
3. Explorer's Arcane Pack (page 5) - adventuring gear that includes ink, parchment, component pouch, rations, 50ft silk rope, and herbalist kit.

As for their official 5e mechanics, the context does not provide explicit rules for the magic items. However, based on the descriptions, we can make some inferences:

1. Codex of Starlight (page 5) - contains spells, but no specific mechanics are mentioned.
2. Ring of Selûne's Grace (page 5) - no specific mechanics are mentioned, but it glows softly when fiends or undead approach within 60 feet.
3. Explorer's Arcane Pack (page 5) - no specific mechanics are mentioned, b

### D4 — 🔬 EXPERIMENT: Combining Both (Hybrid Retrieval)

The most powerful approach is to merge results from both your local index AND the web,  
then let the LLM synthesise across all sources. This is how production systems like Perplexity work.

This cell is **fully written** — just run it and observe how the answer changes when the LLM has both local and web context.

In [41]:
HYBRID_SYSTEM_PROMPT = """You are a research assistant with access to both a local document library and live web search results.
You will receive context from two sources: LOCAL DOCUMENT and WEB SEARCH.
Synthesise information from both sources to give the most complete and accurate answer.
Clearly label which source each piece of information comes from:
  - Local: cite the page number
  - Web: cite the URL
If the sources contradict each other, note the discrepancy and explain which seems more current."""

def ask_hybrid_rag(question: str, local_retriever, k_web: int = 3) -> str:
    """Retrieves from both local ChromaDB and web, then generates a combined answer."""
    # Get local chunks
    local_docs = local_retriever.invoke(question)
    # Get web results
    web_docs   = web_search_to_docs(question)

    # Format each source separately so the LLM can attribute correctly
    local_context = format_context(local_docs)
    web_context   = format_context(web_docs)

    prompt = f"""Question: {question}

=== LOCAL DOCUMENT CONTEXT ===
{local_context}

=== LIVE WEB SEARCH CONTEXT ===
{web_context}

Synthesise both sources to answer the question. Label each fact with its source (page number or URL)."""

    client   = Groq(api_key=os.getenv("GROQ_API_KEY"))
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": HYBRID_SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

# Run with the same question from D3
hybrid_retriever = store_800.as_retriever(search_kwargs={"k": 3})
hybrid_answer    = ask_hybrid_rag(comparison_question, hybrid_retriever)

print("=" * 60)
print("🔀 HYBRID RAG (local PDF + live web)")
print("=" * 60)
print(hybrid_answer)

🔀 HYBRID RAG (local PDF + live web)
Based on the provided sources, Aelion's magic items are:

1. Codex of Starlight (LOCAL DOCUMENT, page 5) - a spellbook containing 12 First-Level and 4 Second-Level spells in elven shorthand.
2. Ring of Selûne's Grace (LOCAL DOCUMENT, page 5) - a ring that glows softly when fiends or undead approach within 60 feet.
3. Explorer's Arcane Pack (LOCAL DOCUMENT, page 5) - a pack of adventuring gear that includes ink, parchment, component pouch, rations, 50ft silk rope, and herbalist kit.

There is no information in the provided sources about Aelion owning any of the magic items listed in the web search results.

Regarding the official 5e mechanics of these items, the Codex of Starlight is not a standard 5e item, so its mechanics are not defined. The Ring of Selûne's Grace is not a standard 5e item, so its mechanics are not defined. The Explorer's Arcane Pack is not a standard 5e item, so its mechanics are not defined.

However, the web search results provi

### 📝 Reflection D

1. **Local vs Web:** For your chosen question, which source gave a more complete answer?

   *Your answer:* Local RAG provided a much more accurate and complete answer because Aelion's inventory is custom homebrew lore specific to the PDF document that does not exist on the public web.

2. **Hybrid answer:** Did combining both sources produce a noticeably better answer than either alone? What did each source contribute?

   *Your answer:* Yes. Local RAG contributed the exact homebrew items and spells owned by Aelion, while Web RAG contributed broader general 5e game mechanics to complement the custom lore.

3. **When would you choose each approach in a real product?**

   | Scenario | Best approach | Reason |
   |---|---|---|
   | Internal HR policy bot | Local RAG | Policy is private, not on the web |
   | Current news summariser | Web RAG | Requires real-time, up-to-date information from live news outlets. |
   | Product manual Q&A with recent updates | Hybrid RAG | Local RAG handles the core product documentation, while Web RAG fetches recent patch notes or updates. |
   | General research assistant | Hybrid RAG | Combines specific user-uploaded documents with dynamic web facts for comprehensive research. |

4. **What is one risk** of using web-augmented RAG that doesn't exist with local RAG?

   *Your answer:* Web noise and hallucinated relevance, the web search can pull irrelevant or unverified external content that pollutes the LLM's context window and leads to off-topic answers.

---
## 🧪 PART E — RAG Evaluation with RAGAS  *(Optional — Stretch Goal)*

> **This section is optional.** Complete Parts A–D first. If you have time remaining, come back here.

### Why evaluate?

So far you have been evaluating RAG the hard way: reading retrieved chunks and answers manually and forming a gut feeling about quality. That works for a few questions — but in a real product with thousands of queries, you need **automated, reproducible metrics**.

### What is RAGAS?

**RAGAS** (Retrieval Augmented Generation Assessment) is an open-source Python library that automatically scores your RAG pipeline on four key metrics — using an LLM as a judge. It requires no human-labelled ground truth for the basic metrics.

| Metric | What it asks | Score range |
|---|---|---|
| **Faithfulness** | Is every claim in the answer supported by the retrieved context? (no hallucination) | 0–1 (higher = better) |
| **Answer Relevancy** | Does the answer actually address the user's question? | 0–1 (higher = better) |
| **Context Recall** | Did retrieval find all the evidence needed to answer? | 0–1 (higher = better) |
| **Context Precision** | Are the retrieved chunks relevant? Low = noisy retrieval | 0–1 (higher = better) |

We will run **Faithfulness** and **Answer Relevancy** — the two that don't require ground-truth reference answers, making them the easiest to use.

### E0 — Install RAGAS

In [ ]:
import sys
!{sys.executable} -m pip install -q ragas datasets
print("✅ RAGAS installed")

### E1 — Guided: Understand the RAGAS Input Format

RAGAS expects a dataset where each row represents one Q&A exchange and contains:

| Field | Type | Description |
|---|---|---|
| `user_input` | `str` | The question that was asked |
| `response` | `str` | The LLM's answer |
| `retrieved_contexts` | `list[str]` | The raw text of each retrieved chunk (as plain strings) |
| `reference` | `str` | *(Optional)* Ground-truth answer — needed for Context Recall but not Faithfulness/Relevancy |

The cell below shows how to build this dataset from your existing `ask_rag()` function.

In [ ]:
# This cell is fully written — run it to see what RAGAS data looks like

# We'll run 3 test questions through our local RAG pipeline
# and collect (question, answer, retrieved_chunks) for each

eval_retriever = store_800.as_retriever(search_kwargs={"k": 4})

# Define a few representative test questions for your document
# 📌 Change these to questions that make sense for YOUR PDF
test_questions = [
    "What are the main topics covered in this document?",
    "What is the current price of Bitcoin?",         # out-of-scope — should score low on faithfulness
    "What does the document say about deadlines?",   # replace with something relevant to your PDF
]

# Collect results
eval_rows = []
for q in test_questions:
    answer, docs = ask_rag(q, eval_retriever)
    eval_rows.append({
        "user_input"          : q,
        "response"            : answer,
        "retrieved_contexts"  : [doc.page_content for doc in docs],  # plain strings, not Document objects
    })
    print(f"✅ Collected: {q[:60]}...")

print(f"\n📊 {len(eval_rows)} rows ready for evaluation")
print()
print("Sample row (first question):")
print(f"  user_input         : {eval_rows[0]['user_input']}")
print(f"  response (first 100): {eval_rows[0]['response'][:100]}...")
print(f"  retrieved_contexts : {len(eval_rows[0]['retrieved_contexts'])} chunks")

### E2 — ✏️ YOUR TURN: Run the RAGAS Evaluation

Now you will build the RAGAS dataset and run evaluation. Fill in the three gaps marked `# ✏️ YOUR TURN`.

**What to expect:**
- The evaluation takes ~30–60 seconds — RAGAS uses an LLM internally to score each answer
- By default RAGAS uses OpenAI. We configure it to use Groq instead (free)
- Scores will be between 0 and 1 — higher is better
- The out-of-scope Bitcoin question should score **low** on faithfulness (the model likely hallucinated)

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

# ── Configure RAGAS to use Groq + local embeddings (both free) ─────────────────
ragas_llm = LangchainLLMWrapper(ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY"),
))
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)

# ── ✏️ YOUR TURN 1: Build the Hugging Face Dataset from eval_rows ──────────────
# Hint: Dataset.from_list(eval_rows)
eval_dataset = ___________  # YOUR CODE HERE

print(f"Dataset columns : {eval_dataset.column_names}")
print(f"Dataset rows    : {len(eval_dataset)}")
print()

# ── ✏️ YOUR TURN 2: Run the evaluation ─────────────────────────────────────────
# Hint: evaluate(
#     dataset=eval_dataset,
#     metrics=[faithfulness, answer_relevancy],
#     llm=ragas_llm,
#     embeddings=ragas_embeddings,
# )
print("Running RAGAS evaluation (takes ~30–60 seconds)...")
results = ___________  # YOUR CODE HERE

print("\n✅ Evaluation complete!")
print()

# ── ✏️ YOUR TURN 3: Print the results as a readable table ──────────────────────
# Hint: results.to_pandas() gives you a DataFrame
# Print columns: user_input, faithfulness, answer_relevancy
df = ___________  # YOUR CODE HERE — convert results to a DataFrame

# Print per-question scores
print(f"{'Question':<50}  {'Faithful':>9}  {'Relevant':>9}")
print("-" * 72)
for _, row in df.iterrows():
    q_short  = str(row.get('user_input', ''))[:48]
    faith    = row.get('faithfulness',    float('nan'))
    relevant = row.get('answer_relevancy', float('nan'))
    print(f"{q_short:<50}  {faith:>9.3f}  {relevant:>9.3f}")

# Print average scores
print("-" * 72)
print(f"{'AVERAGE':<50}  {df['faithfulness'].mean():>9.3f}  {df['answer_relevancy'].mean():>9.3f}")

### 📝 Reflection E

1. **Faithfulness score for the out-of-scope question** (Bitcoin price): was it low as expected? What does a low faithfulness score indicate?

   *Your answer:*

2. **chunk_size=300 vs chunk_size=800:** did the RAGAS scores change? Which metric changed more — faithfulness or relevancy? Why do you think that is?

   *Your answer:*

3. **Limitation of this evaluation setup:** we are using an LLM (LLaMA via Groq) as both the RAG generator AND the RAGAS judge. What problem could this cause?

   *Your answer:*

4. **In a production RAG system**, how often would you run RAGAS evaluation — and on how many questions?

   *Your answer:*

---
## 🧹 Cleanup

Run this cell when you are done to remove all temporary ChromaDB index folders.

In [3]:
import shutil
from pathlib import Path

dirs_to_clean = [
    "./chroma_lab_notebook",
    "./chroma_exp_300",
    "./chroma_exp_800",
    "./chroma_exp_1500",
    "./chroma_k_exp",
    "./chroma_my_pdf",
]
for d in dirs_to_clean:
    if Path(d).exists():
        shutil.rmtree(d)
        print(f"🗑  Removed {d}")
print("✅ Cleanup done")

✅ Cleanup done
